# Smart MCQ Solver — Deep Learning & Generative AI

Portfolio version of the IIT Madras BS Data Science project developed for the Smart MCQ Solver Kaggle competition.

**Task:** Rank the top 3 answer choices (A–E) for each multiple-choice question. The competition metric is MAP@3.

This notebook documents the main approaches explored: a TF-IDF + PyTorch neural network, Sentence-Transformer semantic similarity with MiniLM, and MPNet.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

def create_text(df):
    return (
        "Question: " + df["prompt"].fillna("") +
        " A: " + df["A"].fillna("") +
        " B: " + df["B"].fillna("") +
        " C: " + df["C"].fillna("") +
        " D: " + df["D"].fillna("") +
        " E: " + df["E"].fillna("")
    )

train["text"] = create_text(train)
test["text"] = create_text(test)

label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

print(train[["text", "label"]].head())


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train["text"],
    train["label"],
    test_size=0.2,
    random_state=42,
    stratify=train["label"]
)

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train = vectorizer.fit_transform(X_train)
X_val = vectorizer.transform(X_val)
X_test = vectorizer.transform(test["text"])

print("Training Features:", X_train.shape)
print("Validation Features:", X_val.shape)
print("Test Features:", X_test.shape)


In [ ]:
X_train = X_train.toarray()
X_val = X_val.toarray()
X_test = X_test.toarray()

X_train_tensor = torch.FloatTensor(X_train)
X_val_tensor = torch.FloatTensor(X_val)
X_test_tensor = torch.FloatTensor(X_test)

y_train_tensor = torch.LongTensor(y_train.values)
y_val_tensor = torch.LongTensor(y_val.values)

class MCQDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

train_dataset = MCQDataset(X_train_tensor, y_train_tensor)
val_dataset = MCQDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

class MCQNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 5)
        )

    def forward(self, x):
        return self.network(x)

model = MCQNet(X_train.shape[1])

print(model)


In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    correct = 0
    total = 0
    

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)

        correct += (predicted == y_batch).sum().item()

    accuracy = correct / total

    print(f"Epoch {epoch+1}")
    print(f"Loss : {total_loss:.4f}")
    print(f"Accuracy : {accuracy:.4f}")

    "Epoch": epoch + 1,

    "Training Loss": total_loss,

    "Training Accuracy": accuracy

})


In [ ]:
model.eval()

test_dataset = MCQDataset(X_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

predictions = []

with torch.no_grad():

    for X_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probs = torch.softmax(outputs, dim=1)

        top3 = torch.topk(probs, k=3, dim=1).indices.cpu().numpy()

        predictions.extend(top3)

reverse_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

prediction_strings = []

for row in predictions:
    prediction_strings.append(
        " ".join(reverse_map[x] for x in row)
    )

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": prediction_strings
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("submission.csv created successfully!")


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


In [ ]:
import time

start = time.time()

letters = ["A", "B", "C", "D", "E"]

predictions = []

for _, row in test.iterrows():

    prompt = str(row["prompt"])

    options = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    prompt_embedding = model.encode([prompt])

    option_embeddings = model.encode(options)

    scores = cosine_similarity(
        prompt_embedding,
        option_embeddings
    )[0]

    top3 = np.argsort(scores)[::-1][:3]

    prediction = " ".join(
        letters[i] for i in top3
    )

    predictions.append(prediction)

end = time.time()

inference_time = end - start


In [ ]:
correct = 0

for _, row in train.iterrows():

    prompt = str(row["prompt"])

    options = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    prompt_embedding = model.encode([prompt])

    option_embeddings = model.encode(options)

    scores = cosine_similarity(
        prompt_embedding,
        option_embeddings
    )[0]

    pred = np.argmax(scores)

    if letters[pred] == row["answer"]:
        correct += 1

accuracy = correct / len(train)

print("Accuracy:", accuracy)


In [ ]:
"Accuracy": accuracy,

    "Inference_Time": inference_time,

    "Model": "all-MiniLM-L6-v2"

})


In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import wandb
import time


In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [ ]:
model = SentenceTransformer(
    "all-mpnet-base-v2"
)


In [ ]:
letters = ["A","B","C","D","E"]

predictions = []

start = time.time()

for _, row in test.iterrows():

    prompt = str(row["prompt"])

    options = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    prompt_embedding = model.encode([prompt])

    option_embeddings = model.encode(options)

    scores = cosine_similarity(
        prompt_embedding,
        option_embeddings
    )[0]

    top3 = np.argsort(scores)[::-1][:3]

    prediction = " ".join(
        letters[i]
        for i in top3
    )

    predictions.append(prediction)

end = time.time()

inference_time = end-start


In [ ]:
correct = 0

for _, row in train.iterrows():

    prompt = str(row["prompt"])

    options = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    prompt_embedding = model.encode([prompt])

    option_embeddings = model.encode(options)

    scores = cosine_similarity(
        prompt_embedding,
        option_embeddings
    )[0]

    pred = np.argmax(scores)

    if letters[pred] == row["answer"]:
        correct += 1

accuracy = correct / len(train)

print("Accuracy:", accuracy)


In [ ]:
submission = pd.DataFrame({

    "id":test["id"],

    "prediction":predictions

})

submission.to_csv(

    "submission.csv",

    index=False

)

submission.head()
